In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="6AnkOd4nZEBK1A5KaAfe")
project = rf.workspace("maram-4luwz").project("hand-drawing-all-waoss-o1yf9")
version = project.version(2)
dataset = version.download("folder")

In [ ]:
import os
import shutil

# Define your base path
base_path = "/kaggle/working/Hand-Drawing-All-2" 

# The target folder where everything will end up
target_train_dir = os.path.join(base_path, 'train')

# The folders you want to empty into 'train'
folders_to_merge = ['valid', 'test']

def merge_to_train(source_folders, destination):
    for folder_name in source_folders:
        source_path = os.path.join(base_path, folder_name)
        
        if not os.path.exists(source_path):
            print(f"Skipping {folder_name}: Folder not found.")
            continue

        print(f"Merging {folder_name} into train...")

        # Check for subfolders (images/labels)
        subfolders = [f for f in os.listdir(source_path) if os.path.isdir(os.path.join(source_path, f))]
        
        # If there are subfolders like 'images' and 'labels'
        if subfolders:
            for sub in subfolders:
                src_sub = os.path.join(source_path, sub)
                dst_sub = os.path.join(destination, sub)
                
                os.makedirs(dst_sub, exist_ok=True)
                
                for file in os.listdir(src_sub):
                    shutil.move(os.path.join(src_sub, file), os.path.join(dst_sub, file))
        else:
            # If the files are just sitting directly in valid/test
            for file in os.listdir(source_path):
                shutil.move(os.path.join(source_path, file), os.path.join(destination, file))

        # Remove the now-empty source folder
        shutil.rmtree(source_path)

# Run the merge
merge_to_train(folders_to_merge, target_train_dir)

print("Merge complete. All data is now in the 'train' folder.")

In [ ]:
pip install roboflow split-folders

In [ ]:
import roboflow
import splitfolders
import os
import shutil

input_folder = "/kaggle/working/Hand-Drawing-All-2/train" 

# --- 3. STRATIFIED SPLIT ---
# This will create a new folder 'split_dataset' with train, val, and test 
# folders, keeping the images/labels pairs together.
splitfolders.ratio(
    input_folder, 
    output="final_dataset", 
    seed=42, 
    ratio=(.7, .15, .15), # 80% Train, 10% Val, 10% Test
    group_prefix=None, 
    move=False # Use True if you want to move files instead of copying
)

print("Stratified split complete! Your data is in the 'final_dataset' folder.")

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import os

In [ ]:
# --- 1. KAGGLE CONFIGURATION ---
# Change 'your-dataset-name' to match the folder name in /kaggle/input/
DATASET_NAME = "handdrawn-shapes-hds-dataset" 
BASE_PATH = "/kaggle/working/final_dataset"

# Roboflow usually exports with these subfolders
train_dir = os.path.join(BASE_PATH, 'train')
val_dir = os.path.join(BASE_PATH, 'val')
test_dir = os.path.join(BASE_PATH, 'test')

# Hyperparameters
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
EPOCHS_STAGE_1 = 10
EPOCHS_STAGE_2 = 10

In [ ]:
# --- 2. DATA LOADING ---
# Verify paths before loading
if not os.path.exists(train_dir):
    print(f"Error: {train_dir} not found! Check your Data sidebar for the correct name.")
else:
    print(f"Loading data from: {BASE_PATH}")

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode='grayscale',
    label_mode='categorical'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode='grayscale',
    label_mode='categorical'
)

# --- 2. DATA LOADING (ADD THIS LINE) ---
test_dir = os.path.join(BASE_PATH, 'test')

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode='grayscale',
    label_mode='categorical'
)
AUTOTUNE = tf.data.AUTOTUNE
# Optimization
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

class_names = train_ds.class_names
num_classes = len(class_names)
print(f"\nDetected {num_classes} classes: {class_names}")

# Performance optimization for Kaggle GPUs/TPUs
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
def build_inception_model(num_classes):
    img_input = layers.Input(shape=(128, 128, 1))
    
    # Preprocessing: 0-255 to -1 to 1
    x = layers.Rescaling(1./127.5, offset=-1)(img_input)
    
    # Adapter: 1 to 3 channels
    x = layers.Conv2D(3, (1, 1), padding='same')(x)
    
    base_model = tf.keras.applications.InceptionV3(
        input_shape=(128, 128, 3),
        include_top=False,
        weights='imagenet'
    )
    
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    return models.Model(img_input, outputs), base_model

model, base_model = build_inception_model(num_classes)

In [ ]:
# --- 4. STAGE 1: TRAINING THE HEAD ---
print("\n--- Starting Stage 1: Warm-up (Training Head Only) ---")
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_STAGE_1
)

In [ ]:
# --- 5. STAGE 2: FINE-TUNING THE BODY ---
print("\n--- Starting Stage 2: Fine-tuning (Unfreezing Backbone) ---")
base_model.trainable = True

# Very low learning rate to protect pre-trained weights
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5), 
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_STAGE_2
)

In [ ]:
# --- 6. SAVE OUTPUT ---
# On Kaggle, save to the working directory so you can download it
model.save("/kaggle/working/Hand-Drawing-All-2/train/weight/hand_drawing_Inception_model.h5")
print("\nModel saved to /kaggle/working/final_dataset/hand_drawing_Inception_model.h5")

In [ ]:
import numpy as np
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# 1. Get all predictions and true labels from the test set
print("Generating Confusion Matrix...")
all_preds = []
all_labels = []

# Loop through the test dataset to collect all data
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    all_preds.extend(np.argmax(preds, axis=1))
    all_labels.extend(np.argmax(labels, axis=1))

# 2. Create the Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)

# 3. Plotting using Seaborn
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, 
            yticklabels=class_names)

plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - Hand Drawing Classification')
plt.show()

# 4. Print Classification Report for F1-Score and Recall
print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds, target_names=class_names))

In [ ]:
# 1. Zero out the diagonal (we don't want to see correct predictions)
cm_no_diag = cm.copy()
np.fill_diagonal(cm_no_diag, 0)

# 2. Get the indices of the top 20 misclassifications
# We flatten the matrix, find the indices of the largest values, 
# and then convert those back to 2D coordinates (row, col)
top_20_indices = np.unravel_index(
    np.argsort(cm_no_diag.ravel())[-20:][::-1], 
    cm.shape
)

# 3. Print the results in a readable format
print(f"{'#':<3} | {'Actual Class':<20} | {'Predicted As':<20} | {'Mistakes':<8}")
print("-" * 60)

for i in range(20):
    actual_idx = top_20_indices[0][i]
    predicted_idx = top_20_indices[1][i]
    count = cm[actual_idx, predicted_idx]
    
    # Only print if there are actually mistakes (count > 0)
    if count > 0:
        actual_name = class_names[actual_idx]
        pred_name = class_names[predicted_idx]
        print(f"{i+1:<3} | {actual_name:<20} | {pred_name:<20} | {count:<8}")

# 4. Bonus: Summary of the "Most Confused" class
most_confused_idx = np.argmax(np.sum(cm_no_diag, axis=1))
print(f"\nCRITICAL CLASS: '{class_names[most_confused_idx]}' has the most total errors.")

In [ ]:
# 1. Zero out the diagonal to focus only on errors
cm_no_diag = cm.copy()
np.fill_diagonal(cm_no_diag, 0)

# 2. Calculate total errors per class (sum of each row in the error matrix)
# This tells us how many times a specific 'Actual' class was missed
errors_per_class = np.sum(cm_no_diag, axis=1)

# 3. Get the indices of the Top 3 classes with the most errors
top_3_critical_indices = np.argsort(errors_per_class)[-3:][::-1]

print("\n" + "!"*20 + " TOP 3 CRITICAL CLASSES " + "!"*20)
print(f"{'Rank':<6} | {'Class Name':<25} | {'Total Errors':<12} | {'Most Confused With'}")
print("-" * 75)

for rank, idx in enumerate(top_3_critical_indices, 1):
    class_name = class_names[idx]
    total_err = errors_per_class[idx]
    
    # Find which specific class this critical class is confused with most often
    confused_with_idx = np.argmax(cm_no_diag[idx])
    confused_with_name = class_names[confused_with_idx]
    
    print(f"#{rank:<5} | {class_name:<25} | {total_err:<12} | {confused_with_name}")

print("-" * 75)
print("ADVICE: Consider merging these classes or adding more unique samples.")

In [ ]:
# 1. Zero out the diagonal to isolate errors
cm_no_diag = cm.copy()
np.fill_diagonal(cm_no_diag, 0)

# 2. Sum errors per class (row-wise sum shows how often 'Actual' was missed)
errors_per_class = np.sum(cm_no_diag, axis=1)

# 3. Get indices for the Top 10 worst performers
top_10_critical_indices = np.argsort(errors_per_class)[-10:][::-1]

print("\n" + "!"*25 + " TOP 10 CRITICAL CLASSES " + "!"*25)
print(f"{'Rank':<6} | {'Class Name':<25} | {'Total Errors':<12} | {'Primary Confusion'}")
print("-" * 80)

for rank, idx in enumerate(top_10_critical_indices, 1):
    class_name = class_names[idx]
    total_err = errors_per_class[idx]
    
    # Find the specific class this one is confused with most often
    confused_with_idx = np.argmax(cm_no_diag[idx])
    confused_with_name = class_names[confused_with_idx]
    
    # Calculate accuracy for just this class (Recall)
    total_samples = np.sum(cm[idx])
    recall = (cm[idx, idx] / total_samples) * 100 if total_samples > 0 else 0
    
    print(f"#{rank:<5} | {class_name:<25} | {total_err:<12} | {confused_with_name} ({recall:.1f}% Acc)")

print("-" * 80)
print("STRATEGY: Focus your data augmentation on these 10 classes to boost overall accuracy.")

In [ ]:
from IPython.display import FileLink
# Replace 'my_model.h5' with the actual path to your file
FileLink(r'/kaggle/working/Hand-Drawing-All-2/train/weight/hand_drawing_Inception_model.h5')